In [1]:
import sys
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
from jiwer import RemovePunctuation
from dotenv import load_dotenv
import spacy
from tqdm import tqdm


!{sys.executable} -m pip install chat_gpt_asr
from chat_gpt_asr.alignment import *

# read env variables
load_dotenv()
Root = os.getenv("ROOT_PATH")

# load spacy
nlp = spacy.load("en_core_web_sm")

## Read dataset

In [5]:
# read dataset
path = os.path.join(Root, "results/results-dev-set/results_noisy/results_tiny/results_sentence_confidence_tiny/results_GPT-3.5-Turbo_tiny/gpt-3.5-turbo-0125/results_without_sentence_confidence_tiny/corrected_transcriptions_sentence_confidence_tiny.json")

#path = "/home/mnaderi/Documents/thesis/chat-gpt-asr/results/results-dev-set/results_noisy/results_tiny/results_sentence_confidence_tiny/results_GPT-3.5-Turbo_tiny/gpt-3.5-turbo-0125/results_without_sentence_confidence_tiny/corrected_transcriptions_sentence_confidence_tiny.json"

with open(path, "r") as f:
    data = json.load(f)
    
    transcriptions = [RemovePunctuation()(d["asr_transcription"]["text"]).lower().strip() for d in data]
    reference_transcriptions = [RemovePunctuation()(d["reference_transcription"]).lower().strip() for d in data]
    corrected_transcriptions = [RemovePunctuation()(d["corrected_asr_transcription"]).lower().strip() for d in data]

## Parts of speech

In [6]:

def identify_operation(a, l):
    if a != "" and l == "":
        return "D"
    elif a == "" and l != "":
        return "I"
    elif a != "" and l != "" and a != l:
        return "S"
    else:
        return '-'

def identify_edit_type_1(rr, aa, ll):
    edit_types = []
    operations = []
    edits = {"Improved":[],"IntroducedError":[],"LeftCorrect":[],"LeftIncorrect":[]}
    # rr, aa, ll = align3(ref, asr, llm, tokenizer_fn)
    if len(rr) == len(aa) == len(ll):
        for r,a,l in zip(rr,aa,ll):
            operation = identify_operation(a, l)
            operations.append(operation)
            if a == l == r:
                edit_types.append("LeftCorrect")  # left it correct
                edits["LeftCorrect"].append((a,l,r))
            elif a != l and l == r:
                edit_types.append("Improved")  # improve it
                edits["Improved"].append((a,l,r))
            elif a != r and l != r:
                edit_types.append("LeftIncorrect")  # left it incorrect
                edits["LeftIncorrect"].append((a,l,r))
            elif a == r and l != r:
                edit_types.append("IntroducedError")  # introducing an error
                edits["IntroducedError"].append((a,l,r))
    else:
        raise Exception
    return edit_types, edits, operations

def preprocess(ref, asr, llm, tokenizer_fn):

    # list of tokens of aligned ref, asr, and llm 
    ref_aligned, asr_aligned, llm_aligned = align3(ref,asr,llm, tokenizer_fn=tokenizer_fn)

    # identify the edit_types and operations
    edit_types, edits, operations = identify_edit_type_1(ref_aligned, asr_aligned, llm_aligned)
    llm_aligned_str = " ".join([r if r else "*" for r in llm_aligned])
    asr_aligned_str = " ".join([r if r else "*" for r in asr_aligned])
    ref_aligned_str = " ".join([r if r else "*" for r in ref_aligned])
    
    # string of aligned ref, asr, and llm with '' replaced by *
    llm_doc = nlp(llm_aligned_str)
    ref_doc = nlp(ref_aligned_str)
    asr_doc = nlp(asr_aligned_str)
    assert len(llm_doc) == len(asr_doc) == len(ref_doc)

    d = {}
    for i in range(len(asr_doc)):

        operation = operations[i]
        edit_type = edit_types[i]
        a_token = asr_doc[i]
        a_pos = "MASKED" if a_token.text == "*" else a_token.pos_
        
        l_token = llm_doc[i]
        l_pos = "MASKED" if l_token.text == "*" else l_token.pos_

        # identify pos
        if operation == 'I':
            pos = l_pos
        elif operation == 'D':
            pos = a_pos
        elif operation == '-':
            pos = a_pos
        elif operation == 'S':
            pos = a_pos
    
        d[(edit_type, pos)] = d.get((edit_type, pos), 0) + 1
        # print(f"{operation=}, {edit_type=}, {a_pos=} {l_pos=}, {pos=}")

    return d

def tokenizer_fn(s):
    return [token.text for token in nlp(s)]

In [10]:
import multiprocessing as mp

# parallelized approach
def process_triplet(input):
    asr, ref, llm = input
    # process_id = os.getpid()
    # print(f"Process ID: {process_id}, Index: {index}")
    return preprocess(ref, asr, llm, tokenizer_fn)

num_processes = mp.cpu_count()  
with mp.Pool(processes=num_processes) as pool:
    # Map function to distribute tasks across processes
    results = list(
        pool.map(
            process_triplet, 
            zip(transcriptions, reference_transcriptions, corrected_transcriptions)
        )
    )


results

[{('LeftCorrect', 'VERB'): 4,
  ('LeftCorrect', 'ADP'): 4,
  ('LeftCorrect', 'PRON'): 4,
  ('LeftCorrect', 'DET'): 2,
  ('LeftCorrect', 'NOUN'): 3,
  ('LeftCorrect', 'ADJ'): 3,
  ('LeftCorrect', 'CCONJ'): 1,
  ('Improved', 'NOUN'): 1},
 {('LeftCorrect', 'PRON'): 1,
  ('LeftCorrect', 'VERB'): 1,
  ('LeftCorrect', 'ADP'): 2,
  ('LeftCorrect', 'DET'): 2,
  ('LeftCorrect', 'NOUN'): 3,
  ('Improved', 'ADV'): 1},
 {('LeftCorrect', 'ADP'): 2,
  ('LeftCorrect', 'NOUN'): 2,
  ('LeftCorrect', 'PRON'): 3,
  ('LeftCorrect', 'AUX'): 1,
  ('LeftCorrect', 'VERB'): 2,
  ('LeftCorrect', 'ADV'): 2,
  ('LeftCorrect', 'ADJ'): 1},
 {('LeftCorrect', 'PRON'): 9,
  ('LeftCorrect', 'AUX'): 3,
  ('LeftCorrect', 'VERB'): 5,
  ('LeftCorrect', 'DET'): 1,
  ('LeftCorrect', 'NOUN'): 4,
  ('LeftCorrect', 'CCONJ'): 1,
  ('Improved', 'PRON'): 1,
  ('LeftCorrect', 'ADP'): 1,
  ('LeftCorrect', 'ADV'): 1,
  ('Improved', 'VERB'): 1},
 {('LeftCorrect', 'INTJ'): 1,
  ('LeftIncorrect', 'MASKED'): 1,
  ('LeftCorrect', 'VERB'):

In [ ]:

# non-parallelized approach
# tokenizer_fn = lambda s: [token.text for token in nlp(s)]
# results = []
# for asr, ref, llm in tqdm(zip(transcriptions,reference_transcriptions,corrected_transcriptions), 
#                          total=len(transcriptions)):
#     d = preprocess(ref, asr, llm, tokenizer_fn)
#     results.append(d)

In [11]:
accumulated_dict = {}

for dictionary in results:
    for key, value in dictionary.items():
        if key in accumulated_dict:
            accumulated_dict[key] += value
        else:
            accumulated_dict[key] = value

In [94]:
keys, _ = zip(*accumulated_dict.items())
ops, poses = zip(*keys)

df = pd.DataFrame(0, columns=np.unique(ops), index=np.unique(poses))
for (op,pos), val in accumulated_dict.items():
    percnt = pos
    df.loc[pos, op] += val

df

,Improved,IntroducedError,LeftCorrect,LeftIncorrect
ADJ,116,36,2891,408
ADP,162,50,5143,489
ADV,61,26,2676,280
AUX,69,20,3148,351
CCONJ,46,4,1990,201
DET,131,34,4304,525
INTJ,1,0,175,52
MASKED,0,0,0,830
NOUN,522,89,7340,1595
NUM,17,3,230,91


In [95]:
df = df.drop(["MASKED","X","SYM","PUNCT"])#.assign(percentage=lambda df_: df_.sum(1))

df

,Improved,IntroducedError,LeftCorrect,LeftIncorrect
ADJ,116,36,2891,408
ADP,162,50,5143,489
ADV,61,26,2676,280
AUX,69,20,3148,351
CCONJ,46,4,1990,201
DET,131,34,4304,525
INTJ,1,0,175,52
NOUN,522,89,7340,1595
NUM,17,3,230,91
PART,30,9,1326,108


In [97]:
df_1 = (df.div(df.sum(1), axis=0)*100).round(2)
df_1

,Improved,IntroducedError,LeftCorrect,LeftIncorrect
ADJ,3.36,1.04,83.77,11.82
ADP,2.77,0.86,88.00,8.37
ADV,2.00,0.85,87.94,9.20
AUX,1.92,0.56,87.74,9.78
CCONJ,2.05,0.18,88.80,8.97
DET,2.62,0.68,86.18,10.51
INTJ,0.44,0.00,76.75,22.81
NOUN,5.47,0.93,76.89,16.71
NUM,4.99,0.88,67.45,26.69
PART,2.04,0.61,90.02,7.33


In [87]:
# pctn = df.sum(0).sum()
# df_1 = df_1.assign(Percentage=(df.sum(1)/pctn).round(2)*100)

In [98]:
df_1

,Improved,IntroducedError,LeftCorrect,LeftIncorrect
ADJ,3.36,1.04,83.77,11.82
ADP,2.77,0.86,88.00,8.37
ADV,2.00,0.85,87.94,9.20
AUX,1.92,0.56,87.74,9.78
CCONJ,2.05,0.18,88.80,8.97
DET,2.62,0.68,86.18,10.51
INTJ,0.44,0.00,76.75,22.81
NOUN,5.47,0.93,76.89,16.71
NUM,4.99,0.88,67.45,26.69
PART,2.04,0.61,90.02,7.33


In [99]:
df_1.index = df_1.index.map(lambda idx: spacy.explain(idx))

df_1

,Improved,IntroducedError,LeftCorrect,LeftIncorrect
adjective,3.36,1.04,83.77,11.82
adposition,2.77,0.86,88.00,8.37
adverb,2.00,0.85,87.94,9.20
auxiliary,1.92,0.56,87.74,9.78
coordinating conjunction,2.05,0.18,88.80,8.97
determiner,2.62,0.68,86.18,10.51
interjection,0.44,0.00,76.75,22.81
noun,5.47,0.93,76.89,16.71
numeral,4.99,0.88,67.45,26.69
particle,2.04,0.61,90.02,7.33


In [100]:
print(df_1.to_latex(float_format="{:.2f}".format))

\begin{tabular}{lrrrr}
\toprule
 & Improved & IntroducedError & LeftCorrect & LeftIncorrect \\
\midrule
adjective & 3.36 & 1.04 & 83.77 & 11.82 \\
adposition & 2.77 & 0.86 & 88.00 & 8.37 \\
adverb & 2.00 & 0.85 & 87.94 & 9.20 \\
auxiliary & 1.92 & 0.56 & 87.74 & 9.78 \\
coordinating conjunction & 2.05 & 0.18 & 88.80 & 8.97 \\
determiner & 2.62 & 0.68 & 86.18 & 10.51 \\
interjection & 0.44 & 0.00 & 76.75 & 22.81 \\
noun & 5.47 & 0.93 & 76.89 & 16.71 \\
numeral & 4.99 & 0.88 & 67.45 & 26.69 \\
particle & 2.04 & 0.61 & 90.02 & 7.33 \\
pronoun & 1.56 & 0.51 & 89.90 & 8.04 \\
proper noun & 6.50 & 0.54 & 48.84 & 44.12 \\
subordinating conjunction & 1.43 & 0.75 & 90.16 & 7.67 \\
verb & 3.98 & 0.96 & 81.24 & 13.82 \\
\bottomrule
\end{tabular}



In [108]:
pctn = df.sum(1).sum()
sum_of_modifications = df.sum(1)
pctn, sum_of_modifications

(52111,
 ADJ      3451
 ADP      5844
 ADV      3043
 AUX      3588
 CCONJ    2241
 DET      4994
 INTJ      228
 NOUN     9546
 NUM       341
 PART     1473
 PRON     7317
 PROPN    1292
 SCONJ    1473
 VERB     7280
 dtype: int64)

In [121]:
df_2 = (df.div(df.sum(0)).assign(Percentage=sum_of_modifications/pctn)*100).round(2)
df_2

,Improved,IntroducedError,LeftCorrect,LeftIncorrect,Percentage
ADJ,6.97,9.09,6.62,6.40,6.62
ADP,9.74,12.63,11.78,7.67,11.21
ADV,3.67,6.57,6.13,4.39,5.84
AUX,4.15,5.05,7.21,5.50,6.89
CCONJ,2.76,1.01,4.56,3.15,4.30
DET,7.87,8.59,9.85,8.23,9.58
INTJ,0.06,0.00,0.40,0.82,0.44
NOUN,31.37,22.47,16.81,25.01,18.32
NUM,1.02,0.76,0.53,1.43,0.65
PART,1.80,2.27,3.04,1.69,2.83


In [124]:
df_2.index = df_2.index.map(lambda idx: spacy.explain(idx))
print(df_2.to_latex(float_format='{:0.2f}'.format))

\begin{tabular}{lrrrrr}
\toprule
 & Improved & IntroducedError & LeftCorrect & LeftIncorrect & Percentage \\
\midrule
adjective & 6.97 & 9.09 & 6.62 & 6.40 & 6.62 \\
adposition & 9.74 & 12.63 & 11.78 & 7.67 & 11.21 \\
adverb & 3.67 & 6.57 & 6.13 & 4.39 & 5.84 \\
auxiliary & 4.15 & 5.05 & 7.21 & 5.50 & 6.89 \\
coordinating conjunction & 2.76 & 1.01 & 4.56 & 3.15 & 4.30 \\
determiner & 7.87 & 8.59 & 9.85 & 8.23 & 9.58 \\
interjection & 0.06 & 0.00 & 0.40 & 0.82 & 0.44 \\
noun & 31.37 & 22.47 & 16.81 & 25.01 & 18.32 \\
numeral & 1.02 & 0.76 & 0.53 & 1.43 & 0.65 \\
particle & 1.80 & 2.27 & 3.04 & 1.69 & 2.83 \\
pronoun & 6.85 & 9.34 & 15.06 & 9.22 & 14.04 \\
proper noun & 5.05 & 1.77 & 1.44 & 8.94 & 2.48 \\
subordinating conjunction & 1.26 & 2.78 & 3.04 & 1.77 & 2.83 \\
verb & 17.43 & 17.68 & 13.54 & 15.78 & 13.97 \\
\bottomrule
\end{tabular}



## old method

In [115]:
sequences = [
    #REF, ASR, LLM
    ("short one here", "shoe order one here", "shorts one her"),
    ("I eat salami pizza salami", "I meat a salami pizza salami", "I eat a big pizza"),
    ("their fingers sear me like scorching fire","their fingers see her me like fire","their fingers sear me like scorching fire")
]

In [66]:
ref,asr,llm = sequences[2]
def print_alignment_1(tuplee):
    tuplee = tuple(map(lambda x: [r if r else '***' for r in x ], (tuplee)))
    for t,x in zip(('REF','ASR','LLM'),tuplee):
        print(f"{t}: {' '.join(x)}")
        
print_alignment_1(align3(ref, asr, llm))

REF: their fingers *** sear me like fire
ASR: their fingers see her me like fire
LLM: their fingers *** sear me like fire


In [68]:
' '.join([t.pos_ for t in nlp("their fingers sear me like fire")])

'PRON NOUN VERB PRON ADP NOUN'

## debug

In [154]:
tokenizer_fn = lambda s: [token.text for token in nlp(s)]
ref, asr, llm = sequences[2]
edit_types, edits, operations = identify_edit_type(ref, asr, llm, tokenizer_fn=tokenizer_fn)

In [157]:
# debug
print('ref: ',ref)
print('asr: ', asr)
print('llm: ', llm)
align3(ref,asr,llm, tokenizer_fn=tokenizer_fn)

ref:  their fingers sear me like scorching fire
asr:  their fingers see her me like fire
llm:  their fingers sear me like scorching fire


(['their', 'fingers', '', 'sear', 'me', 'like', 'scorching', 'fire'],
 ['their', 'fingers', 'see', 'her', 'me', 'like', '', 'fire'],
 ['their', 'fingers', '', 'sear', 'me', 'like', 'scorching', 'fire'])

In [158]:
# debug
edits

{'Improved': [('see', '', ''),
  ('her', 'sear', 'sear'),
  ('', 'scorching', 'scorching')],
 'IntroducedError': [],
 'LeftCorrect': [('their', 'their', 'their'),
  ('fingers', 'fingers', 'fingers'),
  ('me', 'me', 'me'),
  ('like', 'like', 'like'),
  ('fire', 'fire', 'fire')],
 'LeftIncorrect': []}

In [159]:
# debug
edit_types

['LeftCorrect',
 'LeftCorrect',
 'Improved',
 'Improved',
 'LeftCorrect',
 'LeftCorrect',
 'Improved',
 'LeftCorrect']

In [160]:
# debug
operations

['-', '-', 'D', 'S', '-', '-', 'I', '-']

In [161]:
ref_al, asr_al, llm_al = align3(ref,asr,llm, tokenizer_fn=tokenizer_fn)
ref_al, asr_al, llm_al

(['their', 'fingers', '', 'sear', 'me', 'like', 'scorching', 'fire'],
 ['their', 'fingers', 'see', 'her', 'me', 'like', '', 'fire'],
 ['their', 'fingers', '', 'sear', 'me', 'like', 'scorching', 'fire'])

In [162]:
llm_al_str = " ".join([r if r else "*" for r in llm_al])
asr_al_str = " ".join([r if r else "*" for r in asr_al])
ref_al_str = " ".join([r if r else "*" for r in ref_al])

ref_al_str, asr_al_str, llm_al_str

('their fingers * sear me like scorching fire',
 'their fingers see her me like * fire',
 'their fingers * sear me like scorching fire')

In [169]:
l_doc = nlp(llm_al_str)
r_doc = nlp(ref_al_str)
a_doc = nlp(asr_al_str)

d = {}

assert len(l_doc) == len(a_doc) == len(r_doc)

for i in range(len(a_doc)):

    a_token = a_doc[i]
    if a_token.text == '*':
        a_pos = "MASEKD"
    else:
        a_pos = a_token.pos_
    
    l_token = l_doc[i]
    if l_token.text == '*':
        l_pos = "MASKED"
    else:
        l_pos = l_token.pos_
    
    operation = operations[i]
    edit_type = edit_types[i]

    if operation == 'I':
        pos = l_pos
    elif operation == 'D':
        pos = a_pos
    elif operation == '-':
        pos = a_pos
    elif operation == 'S':
        pos = a_pos

    
    d[(edit_type, pos)] = d.get((edit_type, pos), 0) + 1
    
    print(f"{operation=}, {edit_type=}, {a_pos=} {l_pos=}, {pos=}")

d

operation='-', edit_type='LeftCorrect', a_pos='PRON' l_pos='PRON', pos='PRON'
operation='-', edit_type='LeftCorrect', a_pos='NOUN' l_pos='NOUN', pos='NOUN'
operation='D', edit_type='Improved', a_pos='VERB' l_pos='MASKED', pos='VERB'
operation='S', edit_type='Improved', a_pos='PRON' l_pos='VERB', pos='PRON'
operation='-', edit_type='LeftCorrect', a_pos='PRON' l_pos='PRON', pos='PRON'
operation='-', edit_type='LeftCorrect', a_pos='ADP' l_pos='ADP', pos='ADP'
operation='I', edit_type='Improved', a_pos='MASEKD' l_pos='VERB', pos='VERB'
operation='-', edit_type='LeftCorrect', a_pos='NOUN' l_pos='NOUN', pos='NOUN'


{('LeftCorrect', 'PRON'): 2,
 ('LeftCorrect', 'NOUN'): 2,
 ('Improved', 'VERB'): 2,
 ('Improved', 'PRON'): 1,
 ('LeftCorrect', 'ADP'): 1}

In [218]:
keys, values = zip(*d.items())
ops, poses = zip(*keys)

df = pd.DataFrame(0, columns=np.unique(ops), index=np.unique(poses))
for (op,pos), value in d.items():
    df.loc[pos, op] += value

df

,Improved,LeftCorrect
ADP,0,1
NOUN,0,2
PRON,1,2
VERB,2,0


In [ ]:
# deletion => pos (asr)
# insert => pos (llm)
# substitute => pos (llm)

In [146]:
align3(ref,asr,llm, tokenizer_fn=tokenizer_fn)

(['their', 'fingers', '', 'sear', 'me', 'like', 'scorching', 'fire'],
 ['their', 'fingers', 'see', 'her', 'me', 'like', '', 'fire'],
 ['their', 'fingers', '', 'sear', 'me', 'like', 'scorching', 'fire'])

In [78]:
poses = [t.pos_ for t in nlp(llm)]
poses

['PRON', 'NOUN', 'VERB', 'PRON', 'ADP', 'NOUN']

In [86]:
len(edit_types), len(poses)

(7, 6)

In [88]:
ref_al, asr_al, llm_al = align3(ref,asr,llm)
ref_al, asr_al, llm_al

(['their', 'fingers', '', 'sear', 'me', 'like', 'fire'],
 ['their', 'fingers', 'see', 'her', 'me', 'like', 'fire'],
 ['their', 'fingers', '', 'sear', 'me', 'like', 'fire'])

In [104]:
llm_al_str = " ".join([r if r else "*" for r in llm_al])
llm_al_str

'their fingers * sear me like fire'

In [105]:
llm_al_str_pos = [t.pos_ for t in nlp(llm_al_str)]
llm_al_str_pos

['PRON', 'NOUN', 'PUNCT', 'VERB', 'PRON', 'ADP', 'NOUN']

In [108]:
result = []
for token, modification in zip(nlp(llm_al_str), edit_types):
    if token.text == "*":
        pos_tag = "MASKED"
    else:
        pos_tag = token.pos_
    result.append((modification, pos_tag, token.text))

result

[('LeftCorrect', 'PRON', 'their'),
 ('LeftCorrect', 'NOUN', 'fingers'),
 ('Improved', 'MASKED', '*'),
 ('Improved', 'VERB', 'sear'),
 ('LeftCorrect', 'PRON', 'me'),
 ('LeftCorrect', 'ADP', 'like'),
 ('LeftCorrect', 'NOUN', 'fire')]

In [107]:
doc = nlp(llm_al_str)
[(token.text, token.pos_) for token in doc]

[('their', 'PRON'),
 ('fingers', 'NOUN'),
 ('*', 'PUNCT'),
 ('sear', 'VERB'),
 ('me', 'PRON'),
 ('like', 'ADP'),
 ('fire', 'NOUN')]

In [151]:
import sys
!{sys.executable} -m pip install -U pip install --upgrade torchaudio torch==2.0.1 flair

INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of transformer-smaller-training-vocab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 1.7 MB/s eta 0:00:001.6 MB/s eta 0:00:01
INFO: pip is looking at multiple versions of requests[socks] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 8.3 MB/s eta 0:00:000m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 MB 16.3 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 19.5 MB/s eta 0:00

In [152]:
from flair.data import Sentence
from flair.models import SequenceTagger

# load tagger
tagger = SequenceTagger.load("flair/pos-english")

# make example sentence
sentence = Sentence(llm_al_str_doc)

# predict NER tags
tagger.predict(sentence)

# print sentence
print(sentence)

# print predicted NER spans
print('The following NER tags are found:')
# iterate over entities and print
for entity in sentence.get_spans('pos'):
    print(entity)


RuntimeError: Failed to import transformers.modeling_utils because of the following error (look up to see its traceback):
Failed to import transformers.generation.utils because of the following error (look up to see its traceback):
No module named 'torch.distributed.checkpoint'

In [7]:
for ref, asr, llm in sequences:
    print_alignment(align3(ref, asr, llm))
    print()

        0      1    2     3
0   short         one  here
1    shoe  order  one  here
2  shorts         one   her

   0     1  2       3      4       5
0  I   eat     salami  pizza  salami
1  I  meat  a  salami  pizza  salami
2  I   eat  a     big  pizza        



In [8]:
#sequences_1 = [
    # REF, ASR, LLM
    #("Ummm I go to school", "Um I go to school", "I go to school"),
#]


In [5]:
path = os.path.join(Root, "results/results-dev-set/results_noisy/results_tiny/results_sentence_confidence_tiny/results_GPT-3.5-Turbo_tiny/gpt-3.5-turbo-0125/results_without_sentence_confidence_tiny/corrected_transcriptions_sentence_confidence_tiny.json")

#path = "/home/mnaderi/Documents/thesis/chat-gpt-asr/results/results-dev-set/results_noisy/results_tiny/results_sentence_confidence_tiny/results_GPT-3.5-Turbo_tiny/gpt-3.5-turbo-0125/results_without_sentence_confidence_tiny/corrected_transcriptions_sentence_confidence_tiny.json"

with open(path, "r") as f:
    data = json.load(f)
    
    transcriptions = [RemovePunctuation()(d["asr_transcription"]["text"]).lower().strip() for d in data]
    reference_transcriptions = [RemovePunctuation()(d["reference_transcription"]).lower().strip() for d in data]
    corrected_transcriptions = [RemovePunctuation()(d["corrected_asr_transcription"]).lower().strip() for d in data]

In [14]:
# [[token.pos_ for token in nlp(tr)] for tr in corrected_transcriptions[:10]]
p, c = np.unique([token.pos_ for tr in corrected_transcriptions for token in nlp(tr)], return_counts=True)
df = pd.DataFrame([c], columns=p)
df

,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,SYM,VERB,X
0,3447,5733,3025,3560,2207,4889,225,9399,328,1463,7254,1244,5,1475,1,7296,4


In [15]:
df = df.drop(columns=["PUNCT","SYM","X"])
df

,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,SCONJ,VERB
0,3447,5733,3025,3560,2207,4889,225,9399,328,1463,7254,1244,1475,7296


In [29]:
# df.columns = [spacy.explain(_) for _ in df.columns]
(df / df.sum(axis=1).item()*100).round(2).T

,0
adjective,6.69
adposition,11.12
adverb,5.87
auxiliary,6.91
coordinating conjunction,4.28
determiner,9.48
interjection,0.44
noun,18.23
numeral,0.64
particle,2.84


In [48]:
import pandas as pd
from io import StringIO

latex_table = """
\\begin{table}[!h]
    \\centering
    \\caption{Language Model Performance by Part of Speech: Improved, IntroducedError LeftCorrect, and LeftIncorrect (\\%) in \\texttt{dev-other} dataset transcribed by the Tiny Whisper model. The summation across parts of speeches is 100.}
    \\label{tab:dev_other_pos_1}
      \\resizebox{\\textwidth}{!}{%
    \\begin{tabular}{lrrrrr}
\\toprule
 & Improved & IntroducedError & LeftCorrect & LeftIncorrect & Percentage \\\\
\\midrule
adjective & 7.81 & 7.57 & 6.65 & 6.61 &6.69 \\\\
adposition & 10.91 & 11.89 & 11.58 & 8.30 & 11.12 \\\\
adverb & 5.72 & 3.51 & 6.07 & 4.75 &5.87 \\\\
auxiliary & 3.30 & 4.86 & 7.23 & 5.77 & 6.91 \\\\
coordinating conjunction & 3.10 & 2.70 & 4.39 & 3.96&4.28 \\\\
determiner & 5.59 & 7.57 & 9.86 & 8.11& 9.48\\\\
interjection & 0.13 & 0.00 & 0.40 & 0.73&0.44 \\\\
noun & 27.14 & 22.16 & 17.01 & 23.72 &18.23\\\\
numeral & 1.14 & 1.08 & 0.53 & 1.14 &0.64\\\\
particle & 2.42 & 1.89 & 3.02 & 1.82 &2.84\\\\
pronoun & 6.60 & 11.08 & 14.96 & 10.32 &14.07\\\\
proper noun & 4.85 & 2.16 & 1.51 & 7.51&2.41 \\\\
subordinating conjunction & 2.15 & 3.51 & 3.03 & 1.94& 2.86 \\\\
verb & 19.12 & 20.00 & 13.74 & 15.34& 14.15\\\\
\\bottomrule
\\end{tabular}
}
\\end{table}
"""

# Extract the data part from the latex table
data_str = latex_table.split('\\begin{tabular}')[1].split('\\end{tabular}')[0]
data_str = data_str.split('\\midrule')[1].split('\\bottomrule')[0].strip()

# Convert the data string into a CSV format
data_str = data_str.replace('\\\\', '\n').replace('&', ',').replace(' ', '')

# Create a DataFrame from the CSV string
data = StringIO(data_str)
df = pd.read_csv(data, sep=',', names=["Part of Speech", "Improved", "IntroducedError", "LeftCorrect", "LeftIncorrect", "Percentage"])
df = df.set_index("Part of Speech")
df

,Improved,IntroducedError,LeftCorrect,LeftIncorrect,Percentage
Part of Speech,,,,,
adjective,7.81,7.57,6.65,6.61,6.69
adposition,10.91,11.89,11.58,8.30,11.12
adverb,5.72,3.51,6.07,4.75,5.87
auxiliary,3.30,4.86,7.23,5.77,6.91
coordinatingconjunction,3.10,2.70,4.39,3.96,4.28
determiner,5.59,7.57,9.86,8.11,9.48
interjection,0.13,0.00,0.40,0.73,0.44
noun,27.14,22.16,17.01,23.72,18.23
numeral,1.14,1.08,0.53,1.14,0.64


In [50]:
df_new = df.div(df['Percentage'], axis=0).round(2)
df_new

,Improved,IntroducedError,LeftCorrect,LeftIncorrect,Percentage
Part of Speech,,,,,
adjective,1.17,1.13,0.99,0.99,1.0
adposition,0.98,1.07,1.04,0.75,1.0
adverb,0.97,0.60,1.03,0.81,1.0
auxiliary,0.48,0.70,1.05,0.84,1.0
coordinatingconjunction,0.72,0.63,1.03,0.93,1.0
determiner,0.59,0.80,1.04,0.86,1.0
interjection,0.30,0.00,0.91,1.66,1.0
noun,1.49,1.22,0.93,1.30,1.0
numeral,1.78,1.69,0.83,1.78,1.0


In [51]:
df_desired = df.astype(str) + " (" + df_new.astype(str) + ")"
df_desired

,Improved,IntroducedError,LeftCorrect,LeftIncorrect,Percentage
Part of Speech,,,,,
adjective,7.81 (1.17),7.57 (1.13),6.65 (0.99),6.61 (0.99),6.69 (1.0)
adposition,10.91 (0.98),11.89 (1.07),11.58 (1.04),8.3 (0.75),11.12 (1.0)
adverb,5.72 (0.97),3.51 (0.6),6.07 (1.03),4.75 (0.81),5.87 (1.0)
auxiliary,3.3 (0.48),4.86 (0.7),7.23 (1.05),5.77 (0.84),6.91 (1.0)
coordinatingconjunction,3.1 (0.72),2.7 (0.63),4.39 (1.03),3.96 (0.93),4.28 (1.0)
determiner,5.59 (0.59),7.57 (0.8),9.86 (1.04),8.11 (0.86),9.48 (1.0)
interjection,0.13 (0.3),0.0 (0.0),0.4 (0.91),0.73 (1.66),0.44 (1.0)
noun,27.14 (1.49),22.16 (1.22),17.01 (0.93),23.72 (1.3),18.23 (1.0)
numeral,1.14 (1.78),1.08 (1.69),0.53 (0.83),1.14 (1.78),0.64 (1.0)


In [53]:
print(df_desired.to_latex())

\begin{tabular}{llllll}
\toprule
 & Improved & IntroducedError & LeftCorrect & LeftIncorrect & Percentage \\
Part of Speech &  &  &  &  &  \\
\midrule
adjective & 7.81 (1.17) & 7.57 (1.13) & 6.65 (0.99) & 6.61 (0.99) & 6.69 (1.0) \\
adposition & 10.91 (0.98) & 11.89 (1.07) & 11.58 (1.04) & 8.3 (0.75) & 11.12 (1.0) \\
adverb & 5.72 (0.97) & 3.51 (0.6) & 6.07 (1.03) & 4.75 (0.81) & 5.87 (1.0) \\
auxiliary & 3.3 (0.48) & 4.86 (0.7) & 7.23 (1.05) & 5.77 (0.84) & 6.91 (1.0) \\
coordinatingconjunction & 3.1 (0.72) & 2.7 (0.63) & 4.39 (1.03) & 3.96 (0.93) & 4.28 (1.0) \\
determiner & 5.59 (0.59) & 7.57 (0.8) & 9.86 (1.04) & 8.11 (0.86) & 9.48 (1.0) \\
interjection & 0.13 (0.3) & 0.0 (0.0) & 0.4 (0.91) & 0.73 (1.66) & 0.44 (1.0) \\
noun & 27.14 (1.49) & 22.16 (1.22) & 17.01 (0.93) & 23.72 (1.3) & 18.23 (1.0) \\
numeral & 1.14 (1.78) & 1.08 (1.69) & 0.53 (0.83) & 1.14 (1.78) & 0.64 (1.0) \\
particle & 2.42 (0.85) & 1.89 (0.67) & 3.02 (1.06) & 1.82 (0.64) & 2.84 (1.0) \\
pronoun & 6.6 (0.47) & 1

In [10]:
for i, (ref, asr, llm) in enumerate(zip(reference_transcriptions, transcriptions, corrected_transcriptions)):
    if i > 2000 and i<2010:
        print_alignment(align3(ref, asr, llm))
        print()

    0        1   2       3    4       5   6    7     8         9   ...    37  \
0  one  morning  as   kanti  was  seated  in  his  boat  cleaning  ...  edge   
1  one  morning  as  gandhi  was  seated  in  his  boat  cleaning  ...  edge   
2  one  morning  as  gandhi  was  seated  in  his  boat  cleaning  ...  edge   

     38   39     40    41         42       43  44   45       46  
0  with  two  white        ducklings  clasped  to  her   breast  
1  with  two  white  tuck      links  clasped  to  her  pressed  
2  with  two  white        ducklings  clasped  to  her   breast  

[3 rows x 47 columns]

    0     1    2    3      4     5    6      7    8        9     10         11
0  the  girl  put  the  birds  into  the  water  and  watched  them  anxiously
1  the  girl  put  the  birds  into  the  water  and    watch  them  anxiously
2  the  girl  put  the  birds  into  the  water  and  watched  them  anxiously

        0      1      2      3    4    5   6    7     8         9    10  \

In [125]:
def identify_operation(r, a, l):
    if r == a == l:
        return "-"
    elif a != "" and l == "":
        return "D"
    elif a == "" and l != "":
        return "I"
    elif a != "" and l != "" and a != l:
        return "S"
    else:
        raise Exception('can not be the case!', r, a, l)

def identify_edit_type(ref, asr, llm, tokenizer_fn=None):
    edit_types = []
    operations = []
    edits = {"Improved":[],"IntroducedError":[],"LeftCorrect":[],"LeftIncorrect":[]}
    rr, aa, ll = align3(ref, asr, llm, tokenizer_fn)
    if len(rr) == len(aa) == len(ll):
        for r,a,l in zip(rr,aa,ll):
            operation = identify_operation(r, a, l)
            operations.append(operation)
            if a == l == r:
                edit_types.append("LeftCorrect")  # left it correct
                edits["LeftCorrect"].append((a,l,r))
            elif a != l and l == r:
                edit_types.append("Improved")  # improve it
                edits["Improved"].append((a,l,r))
            elif a != r and l != r:
                edit_types.append("LeftIncorrect")  # left it incorrect
                edits["LeftIncorrect"].append((a,l,r))
            elif a == r and l != r:
                edit_types.append("IntroducedError")  # introducing an error
                edits["IntroducedError"].append((a,l,r))
    else:
        raise Exception
    return edit_types, edits, operations

In [12]:
tokenizer_fn = lambda s: [token.text for token in nlp(s)]
data_1 = []
data_2 = []
for i, (ref, asr, llm) in enumerate(zip(reference_transcriptions, transcriptions, corrected_transcriptions)):
    #if i > 10:
        #break
    edit_types, edits = identify_edit_type(ref, asr, llm, tokenizer_fn)
    word_counts = {'Improved': 0, 'IntroducedError': 0, 'LeftCorrect': 0, 'LeftIncorrect': 0}
    poses = [token.pos_ for token in nlp(llm)]
    data_2.append(list(zip(edit_types, poses)))
    for edit_type in edit_types:
        word_counts[edit_type] += 1
        
    
    # Create a DataFrame to store word counts by edit type
    data_1.append(word_counts)
    # df_edit_types = pd.DataFrame(word_counts.items(), columns=['Type', 'Count'])
    
    # Display the DataFrame
    # print("asr: {} len:{} \nllm: {} len: {} \nref: {} len: {}".format(asr, len(asr), llm, len(llm), ref, len(ref)))
    # print('edits: ', edits)
    # print(i, df_edit_types , "\n")

In [13]:
pd.DataFrame(data_1)

,Improved,IntroducedError,LeftCorrect,LeftIncorrect
0,1,0,21,0
1,1,0,9,0
2,0,0,13,0
3,2,0,25,0
4,0,0,10,1
...,...,...,...,...
2859,0,0,13,2
2860,0,0,11,0
2861,0,0,20,2
2862,1,0,30,1


In [14]:
import pandas as pd

# Flatten the list of lists
flattened_data = [item for sublist in data_2 for item in sublist]

# Initialize a dictionary to store counts
combination_counts = {}

# Count the occurrences of each unique combination
for operation, pos in flattened_data:
    if (operation, pos) in combination_counts:
        combination_counts[(operation, pos)] += 1
    else:
        combination_counts[(operation, pos)] = 1

# Convert the dictionary into a DataFrame
df = pd.DataFrame(combination_counts.items(), columns=['Combination', 'Count'])

# # Split the 'Combination' column into 'Operation' and 'POS'
df[['Operation', 'POS']] = pd.DataFrame(df['Combination'].tolist(), index=df.index)

# Drop the 'Combination' column
df.drop(columns=['Combination'], inplace=True)

# Pivot the DataFrame to have operations as rows and POS counts as columns
pivot_df = df.pivot_table(index='Operation', columns='POS', values='Count', fill_value=0)

pivot_df.to_csv('final_pos_result.csv')

pivot_df
# # Display the resulting DataFrame
# print(pivot_df)


POS,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,SYM,VERB,X
Operation,,,,,,,,,,,,,,,,,
Improved,116.0,162.0,85.0,49.0,46.0,83.0,2.0,403.0,17.0,36.0,98.0,72.0,0.0,32.0,0.0,284.0,0.0
IntroducedError,28.0,44.0,13.0,18.0,10.0,28.0,0.0,82.0,4.0,7.0,41.0,8.0,0.0,13.0,0.0,74.0,0.0
LeftCorrect,2849.0,4957.0,2601.0,3097.0,1879.0,4221.0,173.0,7285.0,229.0,1295.0,6406.0,648.0,5.0,1297.0,0.0,5884.0,3.0
LeftIncorrect,454.0,570.0,326.0,396.0,272.0,557.0,50.0,1629.0,78.0,125.0,709.0,516.0,0.0,133.0,1.0,1054.0,1.0


In [ ]:
# axis=0 # sum of rows
axis=1 # sum of columns
(pivot_df/pivot_df.sum(axis)*100).round(2)

## debug

In [44]:
data = {
    'Improved': [116.0, 162.0, 85.0, 49.0, 46.0, 83.0, 2.0, 403.0, 17.0, 36.0, 98.0, 72.0, 0.0, 32.0, 0.0, 284.0, 0.0],
    'IntroducedError': [28.0, 44.0, 13.0, 18.0, 10.0, 28.0, 0.0, 82.0, 4.0, 7.0, 41.0, 8.0, 0.0, 13.0, 0.0, 74.0, 0.0],
    'LeftCorrect': [2849.0, 4957.0, 2601.0, 3097.0, 1879.0, 4221.0, 173.0, 7285.0, 229.0, 1295.0, 6406.0, 648.0, 5.0, 1297.0, 0.0, 5884.0, 3.0],
    'LeftIncorrect': [454.0, 570.0, 326.0, 396.0, 272.0, 557.0, 50.0, 1629.0, 78.0, 125.0, 709.0, 516.0, 0.0, 133.0, 1.0, 1054.0, 1.0]
}
df = pd.DataFrame(data, index=['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X'])
df = df.T
df

,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,SYM,VERB,X
Improved,116.0,162.0,85.0,49.0,46.0,83.0,2.0,403.0,17.0,36.0,98.0,72.0,0.0,32.0,0.0,284.0,0.0
IntroducedError,28.0,44.0,13.0,18.0,10.0,28.0,0.0,82.0,4.0,7.0,41.0,8.0,0.0,13.0,0.0,74.0,0.0
LeftCorrect,2849.0,4957.0,2601.0,3097.0,1879.0,4221.0,173.0,7285.0,229.0,1295.0,6406.0,648.0,5.0,1297.0,0.0,5884.0,3.0
LeftIncorrect,454.0,570.0,326.0,396.0,272.0,557.0,50.0,1629.0,78.0,125.0,709.0,516.0,0.0,133.0,1.0,1054.0,1.0


In [51]:
df = df.drop(columns=["X", "SYM", "PUNCT"])

In [62]:
df_column_wise = (df.div(df.sum(axis=1), axis=0)*100).round(2)
df_column_wise
# print(df_column_wise.T.to_latex(float_format="{:.2f}".format))

,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,SCONJ,VERB
Improved,7.81,10.91,5.72,3.30,3.10,5.59,0.13,27.14,1.14,2.42,6.60,4.85,2.15,19.12
IntroducedError,7.57,11.89,3.51,4.86,2.70,7.57,0.00,22.16,1.08,1.89,11.08,2.16,3.51,20.00
LeftCorrect,6.65,11.58,6.07,7.23,4.39,9.86,0.40,17.01,0.53,3.02,14.96,1.51,3.03,13.74
LeftIncorrect,6.61,8.30,4.75,5.77,3.96,8.11,0.73,23.72,1.14,1.82,10.32,7.51,1.94,15.34


In [61]:
df_row_wise = (df.div(df.sum(axis=0), axis=1)*100).round(2)
df_row_wise.T
# print(df_row_wise.T.to_latex(float_format="{:.2f}".format))

,Improved,IntroducedError,LeftCorrect,LeftIncorrect
ADJ,3.37,0.81,82.65,13.17
ADP,2.83,0.77,86.46,9.94
ADV,2.81,0.43,85.98,10.78
AUX,1.38,0.51,86.99,11.12
CCONJ,2.08,0.45,85.14,12.32
DET,1.70,0.57,86.34,11.39
INTJ,0.89,0.00,76.89,22.22
NOUN,4.29,0.87,77.51,17.33
NUM,5.18,1.22,69.82,23.78
PART,2.46,0.48,88.52,8.54


In [17]:
ref = "i don't go to school"
asr = "i not go to school"
ll  = "i don't go to uni all"
align3(ref, asr, ll, tokenizer_fn )

(['i', 'do', "n't", 'go', 'to', '', 'school'],
 ['i', '', 'not', 'go', 'to', '', 'school'],
 ['i', 'do', "n't", 'go', 'to', 'uni', 'all'])

In [1]:
import pandas as pd


In [5]:
df = pd.DataFrame({"A":[1,2,3],"B": [4,5,6]}, index=['a','b','c'])
df

,A,B
a,1,4
b,2,5
c,3,6


In [7]:
sum_of_rows = df.sum(0)
(df/sum_of_rows*100).round(2)

,A,B
a,16.67,26.67
b,33.33,33.33
c,50.00,40.00


## debug Parts of speech

In [11]:
def identify_operation(a, l):
    if a != "" and l == "":
        return "D"
    elif a == "" and l != "":
        return "I"
    elif a != "" and l != "" and a != l:
        return "S"
    else:
        return '-'

def identify_edit_type(ref, asr, llm, tokenizer_fn=None):
    edit_types = []
    operations = []
    edits = {"Improved":[],"IntroducedError":[],"LeftCorrect":[],"LeftIncorrect":[]}
    rr, aa, ll = align3(ref, asr, llm, tokenizer_fn)
    if len(rr) == len(aa) == len(ll):
        for r,a,l in zip(rr,aa,ll):
            operation = identify_operation(a, l)
            operations.append(operation)
            if a == l == r:
                edit_types.append("LeftCorrect")  # left it correct
                edits["LeftCorrect"].append((a,l,r))
            elif a != l and l == r:
                edit_types.append("Improved")  # improve it
                edits["Improved"].append((a,l,r))
            elif a != r and l != r:
                edit_types.append("LeftIncorrect")  # left it incorrect
                edits["LeftIncorrect"].append((a,l,r))
            elif a == r and l != r:
                edit_types.append("IntroducedError")  # introducing an error
                edits["IntroducedError"].append((a,l,r))
    else:
        raise Exception
    return edit_types, edits, operations

In [21]:
from tqdm import tqdm

tokenizer_fn = lambda s: [token.text for token in nlp(s)]

d = {}
for asr, ref, llm in tqdm(zip(transcriptions,reference_transcriptions,corrected_transcriptions), 
                         total=len(transcriptions)):
    
    edit_types, edits, operations = identify_edit_type(ref, asr, llm, tokenizer_fn=tokenizer_fn)
    
    # list of tokens of aligned ref, asr, and llm 
    ref_aligned, asr_aligned, llm_aligned = align3(ref,asr,llm, tokenizer_fn=tokenizer_fn)
    llm_aligned_str = " ".join([r if r else "*" for r in llm_aligned])
    asr_aligned_str = " ".join([r if r else "*" for r in asr_aligned])
    ref_aligned_str = " ".join([r if r else "*" for r in ref_aligned])
    
    # string of aligned ref, asr, and llm with '' replaced by *
    llm_doc = nlp(llm_aligned_str)
    ref_doc = nlp(ref_aligned_str)
    asr_doc = nlp(asr_aligned_str)
    assert len(llm_doc) == len(asr_doc) == len(ref_doc)
    
    for i in range(len(asr_doc)):

        operation = operations[i]
        edit_type = edit_types[i]
        a_token = asr_doc[i]
        a_pos = "MASKED" if a_token.text == "*" else a_token.pos_
        
        l_token = llm_doc[i]
        l_pos = "MASKED" if l_token.text == "*" else l_token.pos_

        # identify pos
        if operation == 'I':
            pos = l_pos
        elif operation == 'D':
            pos = a_pos
        elif operation == '-':
            pos = a_pos
        elif operation == 'S':
            pos = a_pos
    
        d[(edit_type, pos)] = d.get((edit_type, pos), 0) + 1
        # print(f"{operation=}, {edit_type=}, {a_pos=} {l_pos=}, {pos=}")

d

 11%|████████▋                                                                     | 321/2864 [32:10<4:14:51,  6.01s/it]


KeyboardInterrupt: 

In [ ]:
keys, values = zip(*d.items())
ops, poses = zip(*keys)

df = pd.DataFrame(0, columns=np.unique(ops), index=np.unique(poses))
for (op,pos), value in d.items():
    df.loc[pos, op] += value

df